# Multi-provider smoke notebook (batch2) — cantus v0.2.1

**Audience:** release manager (run by hand before tagging v0.2.1).

Exercises the three new Tier 2 adapters shipped in v0.2.1 (Google Gemini, Groq, NVIDIA NIM) end-to-end against the real provider endpoints. The v0.2.0 OpenAI/Anthropic smoke lives in `multi_provider_smoke.ipynb` — also run it for full regression.

**Setup before running:**

```bash
pip install 'cantus[providers]'  # installs openai + anthropic + google-genai + groq
pip install python-dotenv  # secret loader; not a cantus dependency
# Create .env next to this notebook:
#   GOOGLE_API_KEY=...
#   GROQ_API_KEY=...
#   NVIDIA_API_KEY=nvapi-...
```

Note: NVIDIA NIM runs on the OpenAI SDK — there is no `cantus[nvidia]` extras. Google uses the new `google-genai` SDK, not the legacy `google-generativeai`.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import cantus
print('cantus version:', cantus.__version__)
assert cantus.__version__.startswith('0.2.'), 'run against v0.2.x'

## Google Gemini — chat (one round trip)

In [ ]:
from cantus import Message, load_chat_model

google = load_chat_model('google/gemini-2.0-flash')
resp = google.chat([
    Message(role='system', content='Reply in one short sentence.'),
    Message(role='user', content='What is 17 plus 25?'),
])
assert resp.message.content.strip(), 'empty Gemini response'
print(resp.message.content)
print('stop_reason:', resp.stop_reason, '| usage:', resp.usage)

## Google Gemini — stream (text-delta only)

In [ ]:
deltas = list(google.stream([Message(role='user', content='Count 1 to 5.')]))
assert deltas, 'no Gemini stream deltas'
assert all(isinstance(d, str) for d in deltas), 'stream yielded non-str'
print(''.join(deltas))

## Groq — chat (one round trip)

In [ ]:
groq = load_chat_model('groq/llama-3.3-70b-versatile')
resp = groq.chat([
    Message(role='system', content='Reply in one short sentence.'),
    Message(role='user', content='What is 17 plus 25?'),
])
assert resp.message.content.strip(), 'empty Groq response'
print(resp.message.content)
print('stop_reason:', resp.stop_reason, '| usage:', resp.usage)

## Groq — stream (text-delta only)

In [ ]:
deltas = list(groq.stream([Message(role='user', content='Count 1 to 5.')]))
assert deltas, 'no Groq stream deltas'
assert all(isinstance(d, str) for d in deltas), 'stream yielded non-str'
print(''.join(deltas))

## NVIDIA NIM — chat (one round trip)

NIM runs on the OpenAI SDK with `base_url=https://integrate.api.nvidia.com/v1`. The adapter is a thin subclass of `OpenAIChatModel`.

In [ ]:
nvidia = load_chat_model('nvidia/meta/llama-3.3-70b-instruct')
resp = nvidia.chat([
    Message(role='system', content='Reply in one short sentence.'),
    Message(role='user', content='What is 17 plus 25?'),
])
assert resp.message.content.strip(), 'empty NVIDIA response'
print(resp.message.content)
print('stop_reason:', resp.stop_reason, '| usage:', resp.usage)

## NVIDIA NIM — stream (text-delta only)

In [ ]:
deltas = list(nvidia.stream([Message(role='user', content='Count 1 to 5.')]))
assert deltas, 'no NVIDIA stream deltas'
assert all(isinstance(d, str) for d in deltas), 'stream yielded non-str'
print(''.join(deltas))

## All three batch2 providers passed.

Commit this notebook with its outputs (the printed strings) before tagging v0.2.1.